In [2]:
import serial
import struct

FMT = '<H12f3H'

gestures = {
    0: "None",
    1: "Thumbs up",
    2: "Thumbs middle",
    3: "Thumbs down",
    4: "Finger gun",
    5: "BANG"
}

accel_x, accel_y, accel_z, grav_x, grav_y, grav_z, gyro_x, gyro_y, gyro_z, flex_thumb, flex_index, flex_middle, fsr_thumb, fsr_index, fsr_middle = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
gesture = 0

VALID_TOPICS = {0, 1}

buffer = bytearray()

def readTick(ser):
    global accel_x, accel_y, accel_z, grav_x, grav_y, grav_z, gyro_x, gyro_y, gyro_z, flex_thumb, flex_index, flex_middle, fsr_thumb, fsr_index, fsr_middle
    global buffer
    global gesture

    # Read whatever is available (or at least 1 byte)
    chunk = ser.read(64)
    buffer += chunk

    while True:
        # Need at least 2 bytes to check topic
        if len(buffer) < 2:
            break

        # Peek at potential topic
        topic = struct.unpack_from('<H', buffer, 0)[0]

        if topic == 0:
            PACKET_SIZE = 56
        else:
            PACKET_SIZE = 4

        if topic not in VALID_TOPICS:
            # Not aligned → discard 1 byte and try again
            buffer.pop(0)
            continue

        # Now check if we have a full packet
        if len(buffer) < PACKET_SIZE:
            break

        # Extract full packet
        packet_bytes = buffer[:PACKET_SIZE]
        del buffer[:PACKET_SIZE]

        # Unpack
        if topic == 0:
            unpacked = struct.unpack(FMT, packet_bytes)

            topic = unpacked[0]
            accel_x, accel_y, accel_z = unpacked[1:4]
            grav_x, grav_y, grav_z = unpacked[4:7]
            gyro_x, gyro_y, gyro_z  = unpacked[7:10]
            flex_thumb, flex_index, flex_middle = unpacked[10:13]
            fsr_thumb, fsr_index, fsr_middle = unpacked[13:16]
        
        elif topic == 1:
            unpacked = struct.unpack('<HH', packet_bytes)

            topic = unpacked[0]
            gesture = unpacked[1]

            print("Gesture:", gesture)

In [ ]:
# Capture data

ser = serial.Serial('COM8', 115200, timeout=1)

CAPTURE_SECONDS = 5
CAPTURE_RATE = 100

for i in range(CAPTURE_SECONDS * CAPTURE_RATE):
    readTick(ser)
    